In [15]:
import pandas as pd

df = pd.read_csv(
        "/scratch/storageA/zaleski_bulls/bulls-vcf-pipeline/data/ref_panel/Chr24_phased_snp.vcf.gz",
        sep="\t",
        header=13,
        # nrows=1000,
        usecols=["#CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"],
        compression="gzip",
        engine="pyarrow",
    )

df

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT
0,24,997,24_997_A_G,A,G,.,PASS,AN=5952;AC=16,GT
1,24,1025,24_1025_G_A,G,A,.,PASS,AN=5952;AC=26,GT
2,24,1027,24_1027_T_C,T,C,.,PASS,AN=5952;AC=195,GT
3,24,1033,24_1033_C_G,C,G,.,PASS,AN=5952;AC=2773,GT
4,24,1034,24_1034_A_T,A,T,.,PASS,AN=5952;AC=16,GT
...,...,...,...,...,...,...,...,...,...
1580092,24,62305516,24_62305516_G_A,G,A,.,PASS,AN=5952;AC=207,GT
1580093,24,62305655,24_62305655_T_A,T,A,.,PASS,AN=5952;AC=4,GT
1580094,24,62305719,24_62305719_A_T,A,T,.,PASS,AN=5952;AC=12,GT
1580095,24,62305805,24_62305805_G_C,G,C,.,PASS,AN=5952;AC=712,GT


In [16]:
import pysam

# Путь к референсному геному
# reference_genome_path = "data/old_reference/Bos_taurus.UMD3.1.dna.toplevel.fa"


reference_genome_path = "/scratch/storageA/zaleski_bulls/bulls-vcf-pipeline/data/reference/ncbi_dataset/data/GCF_002263795.3/GCF_002263795.3_ARS-UCD2.0_genomic.fna"
fasta_handle = pysam.FastaFile(reference_genome_path)


# Хромомосомы NCBI ARS-UCD2.0
chromosome_map = {
    "1": "NC_037328.1",
    "2": "NC_037329.1",
    "3": "NC_037330.1",
    "4": "NC_037331.1",
    "5": "NC_037332.1",
    "6": "NC_037333.1",
    "7": "NC_037334.1",
    "8": "NC_037335.1",
    "9": "NC_037336.1",
    "10": "NC_037337.1",
    "11": "NC_037338.1",
    "12": "NC_037339.1",
    "13": "NC_037340.1",
    "14": "NC_037341.1",
    "15": "NC_037342.1",
    "16": "NC_037343.1",
    "17": "NC_037344.1",
    "18": "NC_037345.1",
    "19": "NC_037346.1",
    "20": "NC_037347.1",
    "21": "NC_037348.1",
    "22": "NC_037349.1",
    "23": "NC_037350.1",
    "24": "NC_037351.1",
    "25": "NC_037352.1",
    "26": "NC_037353.1",
    "27": "NC_037354.1",
    "28": "NC_037355.1",
    "29": "NC_037356.1",
    "30": "NC_037357.1",
    "31": "NC_082638.1",
    "": "NC_006853.1",  # В данных нету
}

# Хромомосомы NCBI UMD3.1
# chromosome_map = {
#     1: 'AC_000158.1',
#     2: 'AC_000159.1',
#     3: 'AC_000160.1',
#     4: 'AC_000161.1',
#     5: 'AC_000162.1',
#     6: 'AC_000163.1',
#     7: 'AC_000164.1',
#     8: 'AC_000165.1',
#     9: 'AC_000166.1',
#     10: 'AC_000167.1',
#     11: 'AC_000168.1',
#     12: 'AC_000169.1',
#     13: 'AC_000170.1',
#     14: 'AC_000171.1',
#     15: 'AC_000172.1',
#     16: 'AC_000173.1',
#     17: 'AC_000174.1',
#     18: 'AC_000175.1',
#     19: 'AC_000176.1',
#     20: 'AC_000177.1',
#     21: 'AC_000178.1',
#     22: 'AC_000179.1',
#     23: 'AC_000180.1',
#     24: 'AC_000181.1',
#     25: 'AC_000182.1',
#     26: 'AC_000183.1',
#     27: 'AC_000184.1',
#     28: 'AC_000185.1',
#     29: 'AC_000186.1',
#     30: 'AC_000187.1'
# }


def get_ref_base(chromosome: str, position: int) -> str | None:
    position = int(position)
    chromosome = chromosome_map[str(chromosome)]

    base = fasta_handle.fetch(reference=chromosome, start=position - 1, end=position)
    return base.upper()

In [17]:
import pandas as pd

# Предположим, твой DF уже загружен как df
# Если CHROM — это число, переводим в строку для функции
# Если POS — это строка, переводим в int

def check_reference(row):
    try:
        # 1. Получаем базу из референса
        # Обязательно приводим к строке и числу, убираем пробелы
        chrom = str(row['#CHROM']).strip()
        pos = int(row['POS'])
        
        actual_ref = get_ref_base(chrom, pos)
        
        # 2. Если get_ref_base вернула None или ошибку
        if not actual_ref:
            return False
            
        # 3. Сравнение с защитой:
        # - Приводим обе буквы к верхнему регистру (T == t)
        # - Убираем возможные невидимые пробелы .strip()
        vcf_ref = str(row['REF']).strip().upper()
        actual_ref = str(actual_ref).strip().upper()
        
        return vcf_ref == actual_ref
        
    except Exception as e:
        # Если в строке мусор (например, POS — это не число)
        return False
# Применяем проверку (создаем булеву колонку)
# df['is_match'] = df.apply(check_reference, axis=1)

df['position'] = df.apply(lambda row: f"{row['#CHROM']}_{row['POS']}", axis=1)

# Считаем количество совпадений
# matches = df['is_match'].sum()
# total = len(df)

# print(f"Совпало: {matches} из {total} ({matches/total:.2%})")

In [18]:
df_2 = pd.read_csv(
        "/scratch/storageA/zaleski_bulls/bulls-vcf-pipeline/bin/HORUS_shifted_fixed.vcf.gz",
        sep="\t",
        header=34,
        # nrows=100,
        usecols=["#CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"],
        compression="gzip",
        # engine="pyarrow",
    )

df_2

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT
0,1,655346,BovineHD0100000005,A,C,.,PASS,.,GT
1,1,776231,Hapmap43437-BTA-101873,A,G,.,PASS,.,GT
2,1,907810,ARS-BFGL-NGS-16466,C,T,.,PASS,.,GT
3,1,918822,BovineHD0100000079,A,G,.,PASS,.,GT
4,1,930560,BovineHD0100000082,G,A,.,PASS,.,GT
...,...,...,...,...,...,...,...,...,...
58620,31,6758621,BovineHD0800019233,T,C,.,PASS,.,GT
58621,31,7084156,BovineHD3100001414,C,T,.,PASS,.,GT
58622,31,9388211,BovineHD3100001376,G,A,.,PASS,.,GT
58623,31,12978879,BovineHD3100000364,T,G,.,PASS,.,GT


In [19]:
def check_reference(row):
    try:
        # 1. Получаем базу из референса
        # Обязательно приводим к строке и числу, убираем пробелы
        chrom = str(row['#CHROM']).strip()
        pos = int(row['POS'])
        
        actual_ref = get_ref_base(chrom, pos)
        
        # 2. Если get_ref_base вернула None или ошибку
        if not actual_ref:
            return False
            
        # 3. Сравнение с защитой:
        # - Приводим обе буквы к верхнему регистру (T == t)
        # - Убираем возможные невидимые пробелы .strip()
        vcf_ref = str(row['REF']).strip().upper()
        actual_ref = str(actual_ref).strip().upper()
        
        return vcf_ref == actual_ref
        
    except Exception as e:
        # Если в строке мусор (например, POS — это не число)
        return False

# Применяем проверку (создаем булеву колонку)
# df_2['is_match'] = df_2.apply(check_reference, axis=1)
df_2['position'] = df_2.apply(lambda row: f"{row['#CHROM']}_{row['POS']}", axis=1)

# Считаем количество совпадений
# matches = df_2['is_match'].sum()
# total = len(df_2)

# print(f"Совпало: {matches} из {total} ({matches/total:.2%})")

In [20]:
df_2

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,position
0,1,655346,BovineHD0100000005,A,C,.,PASS,.,GT,1_655346
1,1,776231,Hapmap43437-BTA-101873,A,G,.,PASS,.,GT,1_776231
2,1,907810,ARS-BFGL-NGS-16466,C,T,.,PASS,.,GT,1_907810
3,1,918822,BovineHD0100000079,A,G,.,PASS,.,GT,1_918822
4,1,930560,BovineHD0100000082,G,A,.,PASS,.,GT,1_930560
...,...,...,...,...,...,...,...,...,...,...
58620,31,6758621,BovineHD0800019233,T,C,.,PASS,.,GT,31_6758621
58621,31,7084156,BovineHD3100001414,C,T,.,PASS,.,GT,31_7084156
58622,31,9388211,BovineHD3100001376,G,A,.,PASS,.,GT,31_9388211
58623,31,12978879,BovineHD3100000364,T,G,.,PASS,.,GT,31_12978879


In [21]:
merge_df = pd.merge(
    left=df_2[["position", "REF", "ALT"]],
    right=df[["position", "REF", "ALT"]],
    on="position",
    suffixes=("_target", "_ref"),
)

In [22]:
# Проверяем, где REF и ALT не совпадают
mismatch = merge_df[
    (merge_df['REF_target'] != merge_df['REF_ref']) | 
    (merge_df['ALT_target'] != merge_df['ALT_ref'])
]

print(f"Найдено совпадений по позициям: {len(merge_df)}")
print(f"Из них не совпадают по аллелям: {len(mismatch)}")

Найдено совпадений по позициям: 1375
Из них не совпадают по аллелям: 1


In [23]:
df_2[df_2["#CHROM"] == 24]

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,position
49554,24,112601,BovineHD2400000076,T,C,.,PASS,.,GT,24_112601
49555,24,163270,BovineHD2400000069,C,T,.,PASS,.,GT,24_163270
49556,24,300821,Hapmap54014-rs29018901,T,C,.,PASS,.,GT,24_300821
49557,24,335857,BovineHD4100016339,A,G,.,PASS,.,GT,24_335857
49558,24,391651,ARS-BFGL-NGS-103469,A,G,.,PASS,.,GT,24_391651
...,...,...,...,...,...,...,...,...,...,...
50997,24,61832911,BovineHD2400018165,G,T,.,PASS,.,GT,24_61832911
50998,24,61954180,BovineHD2400018199,A,G,.,PASS,.,GT,24_61954180
50999,24,61991351,BTB-01890193,G,A,.,PASS,.,GT,24_61991351
51000,24,62039634,BovineHD2400018253,T,G,.,PASS,.,GT,24_62039634


In [24]:
merge_df.to_csv("merge_df", index=False)

In [25]:
merge_df

,position,REF_target,ALT_target,REF_ref,ALT_ref
0,24_112601,T,C,T,C
1,24_163270,C,T,C,T
2,24_300821,T,C,T,C
3,24_335857,A,G,A,G
4,24_391651,A,G,A,G
...,...,...,...,...,...
1370,24_61832911,G,T,G,T
1371,24_61954180,A,G,A,G
1372,24_61991351,G,A,G,A
1373,24_62039634,T,G,T,G


In [26]:
filtered_df = df[df['position'].isin(df_2["position"])]
filtered_df

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,position
2079,24,112601,24_112601_T_C,T,C,.,PASS,AN=5952;AC=1968,GT,24_112601
3031,24,163270,24_163270_C_T,C,T,.,PASS,AN=5952;AC=2853,GT,24_163270
5653,24,300821,24_300821_T_C,T,C,.,PASS,AN=5952;AC=3860,GT,24_300821
6547,24,335857,24_335857_A_G,A,G,.,PASS,AN=5952;AC=2253,GT,24_335857
7656,24,391651,24_391651_A_G,A,G,.,PASS,AN=5952;AC=3217,GT,24_391651
...,...,...,...,...,...,...,...,...,...,...
1570719,24,61832911,24_61832911_G_T,G,T,.,PASS,AN=5952;AC=1923,GT,24_61832911
1573039,24,61954180,24_61954180_A_G,A,G,.,PASS,AN=5952;AC=4165,GT,24_61954180
1573761,24,61991351,24_61991351_G_A,G,A,.,PASS,AN=5952;AC=5165,GT,24_61991351
1574661,24,62039634,24_62039634_T_G,T,G,.,PASS,AN=5952;AC=2096,GT,24_62039634


In [27]:
import pandas as pd
import os

# 1. Загрузка
file_path = '/scratch/storageA/zaleski_bulls/bulls-vcf-pipeline/data/all_chr_linkage_map_JDS.csv'
df = pd.read_csv(file_path)

os.makedirs('genetic_maps', exist_ok=True)

# 2. Обработка всех хромосом
for chrom_val in df['chr'].unique():
    df_chrom = df[df['chr'] == chrom_val].copy()
    
    # Считаем среднее cM (если нужно)
    df_chrom['avg_cM'] = (df_chrom['male_cM'] + df_chrom['female_cM']) / 2
    
    # 3. ФОРМИРУЕМ СТРОГИЙ PLINK FORMAT:
    # Col 1: Chromosome
    # Col 2: Marker ID (ставим '.')
    # Col 3: Genetic position in cM
    # Col 4: Physical position in bp (ОБЯЗАТЕЛЬНО INTEGER)
    
    output_df = pd.DataFrame({
        'chr': df_chrom['chr'].astype(int),
        'id': '.',
        'cM': df_chrom['avg_cM'],
        'pos': df_chrom['Mb'].astype(int)
    })
    
    # 4. Сохранение (разделитель - табуляция или пробел)
    output_name = f'genetic_maps/chr{chrom_val}.map'
    output_df.to_csv(output_name, sep='\t', header=False, index=False)
    
    if chrom_val == 24:
        print(f"Формат для 24 хромосомы зафиксирован:")
        with open(output_name, 'r') as f:
            # Печатаем первые 3 строки для проверки
            print("".join(f.readlines()[:3]))

print("\nГотово!")

Формат для 24 хромосомы зафиксирован:
24	.	0.0	163270
24	.	0.2525	300821
24	.	0.429	438309


Готово!
